
# Dear Canada — Knowledge Graph Extraction Pipeline (v2, KG-oriented)

This replaces the spatial/character-only extraction pipeline with a
knowledge-graph-oriented one, following *Knowledge Graphs and LLMs in Action*
(Negro, Futia, Kůs, Montagna) chs. 5–6, adapted to a first-person serial diary
rather than an institutional archive. **This is a first pass, expected to
change significantly** after you see it run on the pilot book.

## Design decisions this notebook encodes

- **Nodes**: `character`, `location`, `organization`, `significant_event`,
  `identity_category`. `date` is *not* a node — it's a property
  (`entry_date`, `narrated_date`) on entries and relations, since nothing in
  this project needs to traverse *through* a date node rather than just sort
  or filter by it.
- **Diary entries are nodes** (deferring to the textbook's three-layer
  design: File → Entry → Entity-mentions → resolved KG), not just an edge
  tag. This preserves provenance and lets you ask "what else does this entry
  mention" later, and gives the map a clean per-entry unit to hang location
  and date on.
- **Identity categories are recorded *literally***, exactly as the text
  phrases them, with no normalization or merging at this stage. Normalizing
  "the Irish girl" and "a Papist" and "one of them Irish" into a single
  category is a deliberate, later, human-reviewed step — not something the
  extraction step or this notebook does.
- **Two-tier entity resolution** (per our discussion): within-book identity
  resolution happens *during* extraction via the sliding-window registries
  below (this is what handles "R." / "Rossby"-style coreference, and it's
  final for book-scoped entities like fictional characters). Cross-book
  resolution (locations, significant events, organizations that recur across
  the anthology) is deliberately **deferred** — this notebook exports
  clean, book-level canonical entities, but does not attempt to merge them
  against other books. That merge is a separate, later, corpus-wide pass,
  once you have more than one book's output to merge.
- **Significant events are a closed, hand-curated list**, matched against
  (not invented by) the model. The list below is a **temporary placeholder**
  scoped to this one pilot book (a mid-19th-century Irish emigrant girl,
  Great Famine era) — replace it with your real cross-corpus registry before
  running other books.
- **No grounding/confidence fields.** Every relation carries `entry_id`, so
  you can always trace a relation back to its source text; that's the
  standard of evidence we agreed was sufficient for now.
- **Interpretive vs. factual relations** are distinguished by a static
  lookup on relation *type* (e.g. `grieves_for`, `impacted_by` are flagged
  interpretive), not by a second LLM pass — this avoids the merge/redundancy
  problem of running two extraction passes over the same text while still
  letting you filter or weight interpretive claims differently later.
- **Location is handled by two relation types**, not a location property on
  every relation: `located_in` (stationary presence) and `travels_to`
  (movement, including *intended* destinations — "I pray we reach Grosse
  Île soon" counts). Combined with carry-forward logic (below), this gives
  every entry a resolvable location per character without asking the LLM to
  tag location on every single relation it extracts — which is the piece
  that will eventually drive the "map that progresses over time" use case.


## 0. Setup & configuration

In [2]:

import os
import re
import json
import glob
import time
from dataclasses import dataclass, field, asdict
from pathlib import Path
import anthropic
import pandas as pd

# --- paths -------------------------------------------------------------
DATA_DIR = Path("./data")             # expects one CSV per book
OUT_DIR = Path("./output")
REGISTRY_DIR = OUT_DIR / "registries"
CANDIDATE_DIR = OUT_DIR / "candidate_lists"
REVIEW_DIR = OUT_DIR / "for_review"   # anything flagged rather than decided automatically

for d in (OUT_DIR, REGISTRY_DIR, CANDIDATE_DIR, REVIEW_DIR):
    d.mkdir(parents=True, exist_ok=True)

# --- pilot scope ---------------------------------------------------------
BOOK_CODE = "SofS_entries"   # A Sea of Sorrows (Norah McClintock) — pilot book




In [6]:
from dotenv import load_dotenv

env_path = Path.cwd() / ".env"
print(f"Looking for: {env_path}")
print(f"Found: {env_path.exists()}")

load_dotenv(env_path, override=True)

print("Key loaded:", bool(os.getenv("ANTHROPIC_API_KEY")))

Looking for: /Users/sophiebird/Downloads/Dear Canada/NLP and Data Creation from Anthology/.env
Found: True
Key loaded: True


In [7]:
# --- LLM config ------------------------------------------------------------
# Set your key as an environment variable before running:
#   export ANTHROPIC_API_KEY=sk-ant-...
client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

# Check https://docs.claude.com/en/docs/about-claude/models/overview for the
# current model lineup before running at full-corpus scale.
MODEL_NAME = "claude-sonnet-5"

# How many prior entries (full text) to include as context for the sliding
# window, in addition to the compact state_summary. This is for resolving
# pronouns/references (as in the textbook's "R." / "Rossby" example), not for
# inferring continuity — continuity (location carry-forward, etc.) is
# handled downstream in code.
CONTEXT_WINDOW_ENTRIES = 2


## 1. Load book CSV

Expects columns: `book, chapter_id, entry_id, date, entry_text`. The `date`
column is whatever raw date string you have for the entry (it does not need
to be clean — the LLM will re-derive a normalized date from the entry text
itself in the extraction step, since these are often irregularly formatted
in ways that are hard to regex reliably across 38 books, even though they
tend to be *consistently* formatted within a single book).


In [8]:

def load_book(book_code: str, data_dir: Path = DATA_DIR) -> pd.DataFrame:
    path = data_dir / f"{book_code}.csv"
    df = pd.read_csv(path)
    required_cols = {"book", "chapter_id", "entry_id", "date", "entry_text"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"{path} is missing columns: {missing}")
    # Sort into narrative order. Adjust the sort key if chapter/entry ids
    # aren't naturally sortable as strings (e.g. "ch1", "ch2", ..., "ch10").
    df["_chapter_num"] = df["chapter_id"].str.extract(r"(\d+)").astype(int)
    df["_entry_num"] = df["entry_id"].str.extract(r"e(\d+)$").astype(int)
    df = df.sort_values(["_chapter_num", "_entry_num"]).reset_index(drop=True)
    return df

book_df = load_book(BOOK_CODE)
book_df.head()


,book,chapter_id,entry_id,date,entry_text,_chapter_num,_entry_num
0,SofS,c01,chc01_e1,"April 30, 1847",When Grandda told me all the stories he had ab...,1,1
1,SofS,c02,chc02_e1,"May 1, 1847",I heard Ma crying in the night when everyone e...,2,1
2,SofS,c02,chc02_e2,"May 3, 1847",I am crouched behind a rock at the side of the...,2,2
3,SofS,c02,chc02_e3,"May 6, 1847",We left Ireland yesterday and have reached Liv...,2,3
4,SofS,c02,chc02_e4,"May 7, 1847","From a distance, the stone piers and docks of ...",2,4



## 2. NER recall net

A cheap first pass with spaCy to surface candidate character, location, and
organization surface forms. Purely a *recall* tool — it doesn't decide type
or identity, that happens in review (step 3) and in the LLM extraction
itself. `identity_category` is deliberately **not** NER-seeded: those are
extracted directly, verbatim, by the LLM in the main extraction step, since
there's no dictionary of "words a book might use to describe identity" worth
pre-building.

Install once: `python -m spacy download en_core_web_trf`
(`en_core_web_sm` also works, faster/smaller, somewhat lower recall on
period names).


In [9]:

import spacy

try:
    nlp = spacy.load("en_core_web_trf")
except OSError:
    nlp = spacy.load("en_core_web_sm")

PERSON_LABELS = {"PERSON"}
PLACE_LABELS = {"GPE", "LOC", "FAC"}
ORG_LABELS = {"ORG"}

def ner_pass_book(df: pd.DataFrame) -> dict:
    """Run NER over every entry in a book; return candidate surface-form
    lists (character / location / organization) with counts and an example
    entry_id for each."""
    names, places, orgs = {}, {}, {}
    for _, row in df.iterrows():
        doc = nlp(row["entry_text"])
        for ent in doc.ents:
            if ent.label_ in PERSON_LABELS:
                target = names
            elif ent.label_ in PLACE_LABELS:
                target = places
            elif ent.label_ in ORG_LABELS:
                target = orgs
            else:
                continue
            key = ent.text.strip()
            if not key:
                continue
            if key not in target:
                target[key] = {"count": 0, "example_entry_id": row["entry_id"], "label": ent.label_}
            target[key]["count"] += 1
    return {"candidate_characters": names, "candidate_locations": places, "candidate_organizations": orgs}

def save_candidate_lists(book_code: str, candidates: dict):
    path = CANDIDATE_DIR / f"{book_code}_candidates.json"
    with open(path, "w") as f:
        json.dump(candidates, f, indent=2)
    print(f"Wrote {path} — {len(candidates['candidate_characters'])} character forms, "
          f"{len(candidates['candidate_locations'])} location forms, "
          f"{len(candidates['candidate_organizations'])} organization forms.")
    return path

candidates = ner_pass_book(book_df)
save_candidate_lists("SofS_entries", candidates)


/Users/sophiebird/anaconda3/envs/dearcanadaenv/lib/python3.11/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Wrote output/candidate_lists/SofS_entries_candidates.json — 77 character forms, 29 location forms, 3 organization forms.


PosixPath('output/candidate_lists/SofS_entries_candidates.json')


## 3. Review candidate lists

Same pattern as before: either hand-edit the JSON into a `*_reviewed.json`
file, or use the LLM-assisted first pass below and then hand-review it
before treating it as ground truth. **Do not skip the human review** — this
is the coreference-decision step that's cheap to fix now and expensive to
unwind later once the registries (step 5) are built on top of it.


In [12]:

CANONICALIZE_SYSTEM_PROMPT = """You are cleaning up raw named-entity lists \
extracted from one book of a children's historical fiction diary novel \
(the Dear Canada series). You will be given three flat lists: candidate \
CHARACTER surface forms, candidate LOCATION surface forms, and candidate \
ORGANIZATION surface forms, each with an occurrence count.

Your job: group surface forms that refer to the same real-world entity, and \
flag anything that looks like a false positive (a place mistagged as a \
person, a common noun mistagged as a place, the narrator referring to \
herself in the diary's voice, ships/vehicles mistagged as organizations or \
locations).

Do NOT attempt to canonicalize or group anything related to identity, \
ethnicity, religion, or nationality (e.g. do not create a "the Irish" \
character/group entity) — that is handled separately and deliberately kept \
as literal text elsewhere in this pipeline.

Return ONLY valid JSON, no preamble, in this shape:
{
  "characters": [
    {"suggested_id": "johanna_leary", "canonical_label": "Johanna Leary", "aliases": ["Johanna", "I", "me"], "relation_to_narrator": "self"}
  ],
  "locations": [
    {"suggested_id": "grosse_ile", "canonical_label": "Grosse Île", "aliases": ["Grosse Isle", "the island", "the quarantine station"]}
  ],
  "organizations": [
    {"suggested_id": "...", "canonical_label": "...", "aliases": ["..."], "org_type_hint": "e.g. hospital / religious order / school — or null if unclear"}
  ],
  "likely_false_positives": ["..."]
}
"""

def suggest_canonical_entities(candidates: dict) -> dict:
    user_content = json.dumps({
        "candidate_characters": {k: v["count"] for k, v in candidates["candidate_characters"].items()},
        "candidate_locations": {k: v["count"] for k, v in candidates["candidate_locations"].items()},
        "candidate_organizations": {k: v["count"] for k, v in candidates["candidate_organizations"].items()},
    }, indent=2)

    resp = client.messages.create(
        model=MODEL_NAME,
        max_tokens=8000,
        system=CANONICALIZE_SYSTEM_PROMPT,
        output_config={"effort": "low"},
        messages=[{"role": "user", "content": user_content}],
    )

    if resp.stop_reason == "max_tokens":
      raise RuntimeError(
          "Claude used the entire token limit before producing JSON. "
          "Increase max_tokens (try 12000) or reduce the candidate list."
      )

    # A response can contain a thinking block before its text block.
    text_blocks = [
        block.text
        for block in resp.content
        if block.type == "text"
    ]

    if not text_blocks:
        raise ValueError(
            f"No text was returned. Response blocks: "
            f"{[block.type for block in resp.content]}"
        )

    text = "\n".join(text_blocks).strip()

    # Tolerate an accidental ```json ... ``` wrapper.
    text = re.sub(
        r"^```(?:json)?\s*|\s*```$",
        "",
        text,
        flags=re.IGNORECASE | re.MULTILINE,
    ).strip()

    return json.loads(text)

def save_reviewed_suggestion(book_code: str, suggestion: dict):
    path = CANDIDATE_DIR / f"{book_code}_reviewed_DRAFT.json"
    with open(path, "w") as f:
        json.dump(suggestion, f, indent=2)
    print(f"Wrote DRAFT to {path} — review and rename to *_reviewed.json before use.")
    return path

def load_reviewed_entities(book_code: str) -> dict:
    path = CANDIDATE_DIR / f"{book_code}_reviewed.json"
    with open(path) as f:
        return json.load(f)

suggestion = suggest_canonical_entities(candidates)
save_reviewed_suggestion(BOOK_CODE, suggestion)


Wrote DRAFT to output/candidate_lists/SofS_entries_reviewed_DRAFT.json — review and rename to *_reviewed.json before use.


PosixPath('output/candidate_lists/SofS_entries_reviewed_DRAFT.json')

In [15]:
# (hand-review the _DRAFT file, save as *_reviewed.json, then:)
reviewed = load_reviewed_entities(BOOK_CODE)


## 4. Significant-event registry — **TEMPORARY placeholder**

This is a hand-curated, closed list, seeded narrowly for *this pilot book's*
setting (Ireland → Canada East, 1847, Great Famine emigration). It is
**not** meant to cover the other 37 books — build the real cross-corpus
registry separately once you're past the pilot, and treat everything below
as disposable.

The model will only be allowed to attach `impacted_by` relations to events
on this list (matched by alias). If it thinks an entry references some other
nationally/provincially significant event, it's instructed to report that as
an `unlisted_significant_events` candidate instead of inventing a new node —
those get logged to `output/for_review/` for you to look at by hand.


In [16]:

# TEMPORARY — replace with your real, cross-corpus significant-event registry.
SIGNIFICANT_EVENTS_SEED = [
    {
        "event_id": "great_famine",
        "canonical_label": "The Great Famine (An Gorta Mór)",
        "aliases": ["the famine", "the great hunger", "the hunger", "potato famine", "potato blight", "the blight"],
        "date_range": "1845-1852",
        "note": "The Irish potato famine driving emigration in this book's setting.",
    },
    {
        "event_id": "coffin_ship_emigration_1847",
        "canonical_label": "Coffin-ship emigration (1847 famine crossings)",
        "aliases": ["coffin ship", "coffin ships", "the crossing", "emigrant ship", "emigrant ships"],
        "date_range": "1847",
        "note": "Overcrowded, disease-ridden emigrant vessels of the 1847 famine crossing.",
    },
    {
        "event_id": "1847_typhus_epidemic",
        "canonical_label": "1847 typhus epidemic ('ship fever')",
        "aliases": ["ship fever", "typhus", "the fever", "typhus epidemic", "the sickness"],
        "date_range": "1847",
        "note": "The typhus outbreak among famine emigrants, especially at Grosse Île.",
    },
    {
        "event_id": "grosse_ile_quarantine_1847",
        "canonical_label": "Grosse Île quarantine station operations (1847)",
        "aliases": ["grosse ile", "grosse isle", "the quarantine station", "the fever sheds", "quarantine island"],
        "date_range": "1847",
        "note": "The quarantine station and fever sheds on Grosse Île, downriver from Quebec City.",
    },
    {
        "event_id": "assisted_emigration_schemes",
        "canonical_label": "Landlord-assisted emigration schemes",
        "aliases": ["assisted passage", "landlord-assisted emigration", "paid our passage"],
        "date_range": "1840s",
        "note": "Schemes by Irish landlords to pay tenants' passage abroad during the famine.",
    },
    {
        "event_id": "irish_discrimination_in_canada",
        "canonical_label": "Irish discrimination in Canada",
        "aliases": ["irish discrimination", "discrimination against irish", "irish prejudice"],
        "note": "Discrimination faced by Irish immigrants in Canada, during the famine years especially.",
    },
]

@dataclass
class SignificantEventEntry:
    event_id: str
    canonical_label: str
    aliases: list = field(default_factory=list)
    date_range: str = None
    note: str = None

class SignificantEventRegistry:
    """Closed-list only. Never auto-creates new events — resolve() either
    matches an entry on the seed list, or resolve_or_flag() logs an unmatched
    candidate for human review and returns None."""

    def __init__(self):
        self._by_id = {}
        self._alias_to_id = {}
        self.unmatched_candidates = []

    def seed(self, seed_list):
        for e in seed_list:
            entry = SignificantEventEntry(
                event_id=e["event_id"], canonical_label=e["canonical_label"],
                aliases=list(e.get("aliases", [])), date_range=e.get("date_range"), note=e.get("note"),
            )
            self._by_id[entry.event_id] = entry
            for a in entry.aliases + [entry.canonical_label]:
                self._alias_to_id[a.lower()] = entry.event_id

    def resolve(self, label: str):
        if not label:
            return None
        label_l = label.lower().strip()
        if label_l in self._alias_to_id:
            return self._by_id[self._alias_to_id[label_l]]
        # loose substring fallback, since phrasing will vary entry to entry
        for alias, eid in self._alias_to_id.items():
            if alias in label_l or label_l in alias:
                return self._by_id[eid]
        return None

    def resolve_or_flag(self, label: str, entry_id: str, reason: str = ""):
        existing = self.resolve(label)
        if existing:
            return existing
        self.unmatched_candidates.append({"label": label, "entry_id": entry_id, "reason": reason})
        return None

    def export(self) -> list:
        return [asdict(v) for v in self._by_id.values()]



## 5. Registries: character, location, organization

`CharacterRegistry` and `LocationRegistry` carry over the design from the
previous notebook (canonical entity + aliases, maintained in code, updated
as entries are processed in order — this is the within-book resolution
tier). `OrganizationRegistry` is new but follows the same pattern. All three
**do** auto-register genuinely new entities encountered mid-book (unlike the
significant-event registry) — but log them to `pending_review` so you can
check the auto-created ones rather than silently trusting them.

The narrator is seeded into `CharacterRegistry` as an ordinary character
(`relation_to_narrator="self"`) — there's no separate "narrator state"
object. This is what lets `located_in` / `travels_to` carry-forward logic
(step 8) apply identically to her and to everyone else, which is the piece
that will eventually feed a map with more than just the narrator's route on
it.

Identity categories are **not** a registry — see step 9.


In [17]:

@dataclass
class CharacterEntry:
    character_id: str
    canonical_label: str
    aliases: list = field(default_factory=list)
    relation_to_narrator: str = "unknown"   # e.g. self, sister, mother, friend, historical_figure
    first_mentioned_entry: str = None
    last_known_location: str = None
    last_location_stated_entry: str = None

@dataclass
class LocationEntry:
    location_id: str
    canonical_label: str
    aliases: list = field(default_factory=list)
    first_mentioned_entry: str = None
    coordinates: list = None   # filled in a later, separate geocoding pass

@dataclass
class OrganizationEntry:
    org_id: str
    canonical_label: str
    aliases: list = field(default_factory=list)
    org_type_hint: str = None   # free text, e.g. "religious order", "hospital" — not an enforced taxonomy yet
    first_mentioned_entry: str = None


class _AliasRegistryBase:
    """Shared resolve/register_or_flag logic for Character/Location/Organization
    registries, which otherwise differ only in which dataclass they hold."""

    entry_cls = None
    id_field = None

    def __init__(self):
        self._by_id = {}
        self._alias_to_id = {}
        self.pending_review = []

    def seed_from_reviewed(self, reviewed_list, extra_fields=None):
        extra_fields = extra_fields or {}
        for item in reviewed_list:
            kwargs = dict(
                canonical_label=item["canonical_label"],
                aliases=list(item.get("aliases", [])),
                first_mentioned_entry=None,
            )
            kwargs[self.id_field] = item["suggested_id"]
            for k, source_key in extra_fields.items():
                kwargs[k] = item.get(source_key)
            entry = self.entry_cls(**kwargs)
            self._by_id[getattr(entry, self.id_field)] = entry
            for a in entry.aliases + [entry.canonical_label]:
                self._alias_to_id[a.lower()] = getattr(entry, self.id_field)

    def resolve(self, label: str):
        if not label:
            return None
        return self._by_id.get(self._alias_to_id.get(label.lower()))

    def canonical_label(self, label: str) -> str:
        e = self.resolve(label)
        return e.canonical_label if e else label

    def register_or_flag(self, label: str, entry_id: str, **extra_kwargs):
        existing = self.resolve(label)
        if existing:
            return existing
        new_id = re.sub(r"[^a-z0-9]+", "_", label.lower()).strip("_")
        kwargs = dict(canonical_label=label, aliases=[label], first_mentioned_entry=entry_id, **extra_kwargs)
        kwargs[self.id_field] = new_id
        entry = self.entry_cls(**kwargs)
        self._by_id[new_id] = entry
        self._alias_to_id[label.lower()] = new_id
        self.pending_review.append({self.id_field: new_id, "label": label, "entry_id": entry_id})
        return entry

    def export(self) -> list:
        return [asdict(v) for v in self._by_id.values()]


class CharacterRegistry(_AliasRegistryBase):
    entry_cls = CharacterEntry
    id_field = "character_id"

    def seed_from_reviewed(self, reviewed_characters: list):
        super().seed_from_reviewed(reviewed_characters, extra_fields={"relation_to_narrator": "relation_to_narrator"})
        for e in self._by_id.values():
            if e.relation_to_narrator is None:
                e.relation_to_narrator = "unknown"

    def seed_narrator(self, narrator_id: str, narrator_label: str, aliases=None):
        """Ensure the narrator exists as a character even before/without a
        reviewed candidate list, since every book has one."""
        if narrator_id in self._by_id:
            return self._by_id[narrator_id]
        entry = CharacterEntry(character_id=narrator_id, canonical_label=narrator_label,
                                aliases=(aliases or []) + ["I", "me", "my", "myself"],
                                relation_to_narrator="self")
        self._by_id[narrator_id] = entry
        for a in entry.aliases + [entry.canonical_label]:
            self._alias_to_id[a.lower()] = narrator_id
        return entry

    def register_or_flag(self, label: str, entry_id: str, relation_hint: str = "unknown"):
        return super().register_or_flag(label, entry_id, relation_to_narrator=relation_hint)

    def update_location(self, character_id: str, location_label: str, entry_id: str):
        if character_id in self._by_id and location_label:
            self._by_id[character_id].last_known_location = location_label
            self._by_id[character_id].last_location_stated_entry = entry_id


class LocationRegistry(_AliasRegistryBase):
    entry_cls = LocationEntry
    id_field = "location_id"


class OrganizationRegistry(_AliasRegistryBase):
    entry_cls = OrganizationEntry
    id_field = "org_id"

    def seed_from_reviewed(self, reviewed_organizations: list):
        super().seed_from_reviewed(reviewed_organizations, extra_fields={"org_type_hint": "org_type_hint"})



## 6. Identity descriptor log (not a registry)

Per your instruction: identity-related descriptions are recorded **exactly
as the text phrases them**, per character, per entry, with who's doing the
describing. No alias resolution, no merging into categories — that happens
later, by a human, once you can see the full raw vocabulary a book (or the
whole corpus) actually uses.


In [18]:

def new_identity_log():
    return []   # list of dicts: character_id, character_label_at_time, raw_text, attributed_by, entry_id

def log_identity_descriptor(log: list, character_id: str, character_label: str, raw_text: str, attributed_by: str, entry_id: str):
    log.append({
        "character_id": character_id,
        "character_label": character_label,
        "raw_text": raw_text,
        "attributed_by": attributed_by,   # "self" | "narrator" | "other_character" | "unspecified"
        "entry_id": entry_id,
    })



## 7. Basic book context (hand-authored)

Same purpose as before: a short, factual synopsis used only to help the
model disambiguate references (place names, historical allusions, family
relationships) — never as a source of facts about what happens in a
specific entry. Filled in for the pilot book from publicly available
descriptions of *A Sea of Sorrows*.


In [19]:

BASIC_CONTEXT = {
    "SofS": {
        "narrator_id": "johanna_leary",
        "narrator_name": "Johanna Leary",
        "narrator_age_at_writing": 13,
        "setting_summary": (
            "Johanna Leary, 13, emigrates from Ireland during the Great Famine in 1847 with her "
            "parents, older brother Michael (14), and baby brother Patrick, bound for Canada East "
            "(Quebec) aboard a crowded emigrant ('coffin') ship. Typhus ('ship fever') spreads "
            "aboard ship and at the Grosse Île quarantine station near Quebec City; over the "
            "course of the book Johanna loses her baby brother, mother, and father to the "
            "epidemic. She and Michael are separated near the end of the book when he is wrongly "
            "told she has died and leaves to search for their uncle Liam, somewhere in Canada, "
            "leaving Johanna alone."
        ),
        "narrative_structure": "real_time",   # first-person diary, written contemporaneously with events
        "instructions_note": (
            "Use this only to interpret unclear references (place names, historical events "
            "alluded to, family relationships). Do NOT use it to infer facts not stated or "
            "clearly implied in the entry text itself, and do not assume events from the synopsis "
            "have already happened at any given entry — read each entry on its own terms."
        ),
    },
    # "example_book_2": { ... },
}



## 8. State summary

Built entirely in code from the registries — never generated by the model —
and injected into the prompt each turn so the model doesn't have to
re-derive continuity itself. This is also where the closed significant-event
list is surfaced to the model each turn.


In [20]:

def build_state_summary(char_registry: CharacterRegistry, loc_registry: LocationRegistry,
                         org_registry: OrganizationRegistry, sig_event_registry: SignificantEventRegistry) -> dict:
    return {
        "known_characters": [
            {"id": c.character_id, "label": c.canonical_label, "aliases": c.aliases,
             "relation_to_narrator": c.relation_to_narrator, "last_known_location": c.last_known_location}
            for c in char_registry._by_id.values()
        ],
        "known_locations": [
            {"id": l.location_id, "label": l.canonical_label, "aliases": l.aliases}
            for l in loc_registry._by_id.values()
        ],
        "known_organizations": [
            {"id": o.org_id, "label": o.canonical_label, "aliases": o.aliases}
            for o in org_registry._by_id.values()
        ],
        "significant_events_of_interest": [
            {"id": e.event_id, "label": e.canonical_label, "aliases": e.aliases, "date_range": e.date_range}
            for e in sig_event_registry._by_id.values()
        ],
    }



## 9. Extraction schema & prompt

The core of the redesign. One JSON object per entry:

```
{
  "entry_date_raw": "...",            # exactly as it appears in the entry, or null
  "entry_date_normalized": "YYYY-MM-DD" or null,
  "relations": [
    {"subject": "...", "subject_type": "character|location|organization",
     "relation": "works_for|works_with|student_of|studies_with|lives_with|
                  stops_living_with|located_in|travels_to|grieves_for|impacted_by",
     "object": "...", "object_type": "character|location|organization|significant_event",
     "mode": "..."  # optional, travels_to only, only if the text states a means of travel
     "narrated_date": "YYYY-MM-DD" or null}  # only if explicitly different from entry_date
  ],
  "mentions": [ {"surface_form": "...", "type": "character|location|organization"} ],
  "identity_descriptors": [
    {"character": "...", "raw_text": "...", "attributed_by": "self|narrator|other_character|unspecified"}
  ],
  "unlisted_significant_events": [ {"raw_text": "...", "reason": "..."} ]
}
```

Design notes baked into the prompt below:

- `entry_date_normalized` is the model's job (not regex/code), because the
  header phrasing varies across books in ways that are hard to normalize
  generically, even though it's usually *consistent within* one book.
- `narrated_date` is left null in the overwhelming majority of entries (most
  entries narrate "today") — it's only filled when the text is explicitly
  describing something from a different day (a memory, an anticipated
  event). Downstream, null narrated_date defaults to entry_date — the model
  is not asked to do that fill-forward itself.
- `impacted_by` may only target the closed `significant_events_of_interest`
  list injected via the state summary. Anything else goes to
  `unlisted_significant_events` instead of being invented as a new object.
- Identity descriptors must be the literal words used — the prompt
  explicitly forbids substituting a general category for the text's own
  phrasing, with an example of exactly the failure mode you flagged
  (collapsing "the Irish girl" / "a Papist" / etc. into "Indigenous"-style
  over-generalization would be the same mistake here, just a different
  identity).


In [21]:

EXTRACTION_SYSTEM_PROMPT = """You are extracting structured knowledge-graph \
data from one dated entry of a first-person historical-fiction diary novel \
(the Dear Canada series), for an academic project studying spatial and \
identity representation in Canadian children's historical fiction.

You will be given: (1) basic factual context about the book and narrator, \
(2) a compact state summary of characters/locations/organizations/ \
significant-events already known from earlier in this same book, (3) the \
text of the previous {context_window} entries (for resolving pronouns and \
references only — do not report facts from those entries as if they were \
in the current one), and (4) the text of the current entry.

Return ONLY valid JSON, no preamble, no markdown fences, in this exact shape:
{
  "entry_date_raw": "<exactly as it appears in the entry text, or null>",
  "entry_date_normalized": "<YYYY-MM-DD, or null if it cannot be determined>",
  "relations": [
    {"subject": "<surface form>", "subject_type": "character|location|organization",
     "relation": "works_for|works_with|student_of|studies_with|lives_with|stops_living_with|located_in|travels_to|grieves_for|impacted_by",
     "object": "<surface form>", "object_type": "character|location|organization|significant_event",
     "mode": "<only for travels_to, only if explicitly stated, else omit>",
     "narrated_date": "<YYYY-MM-DD, ONLY if explicitly different from entry_date, else null>"}
  ],
  "mentions": [ {"surface_form": "<text>", "type": "character|location|organization"} ],
  "identity_descriptors": [
    {"character": "<surface form of the character being described>",
     "raw_text": "<the exact word(s) the text uses>",
     "attributed_by": "self|narrator|other_character|unspecified"}
  ],
  "unlisted_significant_events": [ {"raw_text": "<what the entry seems to reference>", "reason": "<short reason it seems nationally/provincially significant>"} ]
}

RELATION GUIDANCE:
- "located_in": the subject character is stated as being at/in a place THIS entry, with no sense of ongoing movement.
- "travels_to": movement toward a place — includes departures, arrivals, journeys in progress, AND explicitly stated intentions or hopes to reach a place (e.g. "I pray we reach Grosse Île soon" IS a travels_to relation, object "Grosse Île").
- "impacted_by": the object MUST be one of the significant_events_of_interest given to you (match by id, label, or alias — you may use looser paraphrase than the exact alias list). If the entry seems to reference some other nationally/provincially significant event NOT on that list, do not invent a new object for it — report it under "unlisted_significant_events" instead.
- Only extract a relation the text actually states or clearly implies THIS entry. Leave things out rather than guessing. Continuity/"still true from before" is handled downstream in code, not by you.

IDENTITY DESCRIPTORS — read carefully:
- Extract the LITERAL words the text uses to describe a character's ethnicity, nationality, religion, class, or other identity-related characterization. Copy the phrasing, do not substitute your own category label.
  - Correct: text says "the Irish girl" -> raw_text "the Irish girl" (not "Irish").
  - Correct: text says "we are Papists" -> raw_text "Papists" (not "Catholic").
  - WRONG: generalizing or flattening any group's own specific self-description into a broader external label — this is the single biggest failure mode to avoid, whatever the identity in question.
- Only extract a descriptor when the entry text itself uses identity-related language in that specific entry. Do not tag a character as an identity category just because background context (or the basic_context synopsis) establishes it generally — that's fabricating provenance for something the entry itself never said.

DATE GUIDANCE:
- entry_date_normalized: read the entry's own date heading/text and normalize it to YYYY-MM-DD if the full date (day, month, year) can be determined, using the book's known setting to fill in an ambiguous year only if that is truly unambiguous from context. Otherwise return null — do not guess.
- narrated_date: leave null unless the entry explicitly describes something happening on a DIFFERENT day than the entry's own date (a memory, an anticipated future event). Most entries only narrate "today" and should have narrated_date: null for every relation.

Note common historical-fiction diary conventions: narrators sometimes refer \
to family members by relationship term only ("Da", "Mother", "my brother") \
rather than by name — use the state summary's known_characters to resolve \
these to the correct canonical surface form where possible; if genuinely \
new, just use the surface form as written and it will be registered \
downstream.
"""

def build_user_content(entry_row, context_entries_text, state_summary, basic_context):
    payload = {
        "basic_context": basic_context,
        "state_summary": state_summary,
        "previous_entries_for_reference_only": context_entries_text,
        "current_entry": {
            "entry_id": entry_row["entry_id"],
            "raw_date_column": str(entry_row["date"]),
            "entry_text": entry_row["entry_text"],
        },
    }
    return json.dumps(payload, indent=2)


## 10. Per-entry LLM call

In [ ]:

def call_llm_for_entry(entry_row, prior_entries_df, state_summary, basic_context):
    context_texts = [
        {"entry_id": r["entry_id"], "text": r["entry_text"]}
        for _, r in prior_entries_df.tail(CONTEXT_WINDOW_ENTRIES).iterrows()
    ]
    system_prompt = EXTRACTION_SYSTEM_PROMPT.replace("{context_window}", str(CONTEXT_WINDOW_ENTRIES))
    user_content = build_user_content(entry_row, context_texts, state_summary, basic_context)

    resp = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        #temperature=0,
        system=system_prompt,
        messages=[{"role": "user", "content": user_content}],
    )
    text = resp.content[0].text.strip()
    text = re.sub(r"^```(json)?|```$", "", text, flags=re.MULTILINE).strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError as e:
        return {"_parse_error": str(e), "_raw_text": text}



## 11. Resolution: mapping model output onto the registries

This is where surface forms returned by the model get resolved to stable
IDs. `character` / `location` / `organization` auto-register new entities
(flagged for review); `significant_event` **never** auto-registers — it
either matches the closed list or goes to `unmatched_candidates`.

`INTERPRETIVE_RELATIONS` is the static lookup mentioned above — used at
export time to tag relations, not to change what gets extracted.


In [23]:

INTERPRETIVE_RELATIONS = {"grieves_for", "impacted_by"}

def resolve_entity(label, etype, char_registry, loc_registry, org_registry, sig_event_registry, entry_id):
    if not label:
        return None, None
    if etype == "character":
        e = char_registry.resolve(label) or char_registry.register_or_flag(label, entry_id)
        return e.character_id, e.canonical_label
    if etype == "location":
        e = loc_registry.resolve(label) or loc_registry.register_or_flag(label, entry_id)
        return e.location_id, e.canonical_label
    if etype == "organization":
        e = org_registry.resolve(label) or org_registry.register_or_flag(label, entry_id)
        return e.org_id, e.canonical_label
    if etype == "significant_event":
        e = sig_event_registry.resolve_or_flag(label, entry_id)
        if e is None:
            return None, label   # unresolved: kept as raw label only, no canonical id created
        return e.event_id, e.canonical_label
    return None, label


def process_relation(rel, entry_id, entry_date, char_registry, loc_registry, org_registry, sig_event_registry):
    subj_id, subj_label = resolve_entity(rel.get("subject"), rel.get("subject_type"),
                                          char_registry, loc_registry, org_registry, sig_event_registry, entry_id)
    obj_id, obj_label = resolve_entity(rel.get("object"), rel.get("object_type"),
                                        char_registry, loc_registry, org_registry, sig_event_registry, entry_id)
    narrated_date = rel.get("narrated_date") or entry_date   # default fill-forward happens here, not in the model

    record = {
        "book": BOOK_CODE,
        "entry_id": entry_id,
        "subject_id": subj_id, "subject_label": subj_label, "subject_type": rel.get("subject_type"),
        "relation": rel.get("relation"),
        "object_id": obj_id, "object_label": obj_label, "object_type": rel.get("object_type"),
        "mode": rel.get("mode"),
        "entry_date": entry_date,
        "narrated_date": narrated_date,
        "interpretive": rel.get("relation") in INTERPRETIVE_RELATIONS,
    }

    # Location state update — this is what feeds the map. Applies to ANY
    # character (the narrator included, since she's just another character
    # in the registry), for both located_in and travels_to.
    if rel.get("subject_type") == "character" and rel.get("relation") in ("located_in", "travels_to") and obj_label:
        char_registry.update_location(subj_id, obj_label, entry_id)

    return record


## 12. Process a book (sliding window over entries, in narrative order)

In [24]:

def process_book(book_code: str, df: pd.DataFrame, reviewed: dict, sig_events_seed=SIGNIFICANT_EVENTS_SEED):
    char_registry = CharacterRegistry()
    loc_registry = LocationRegistry()
    org_registry = OrganizationRegistry()
    sig_event_registry = SignificantEventRegistry()

    char_registry.seed_from_reviewed(reviewed.get("characters", []))
    loc_registry.seed_from_reviewed(reviewed.get("locations", []))
    org_registry.seed_from_reviewed(reviewed.get("organizations", []))
    sig_event_registry.seed(sig_events_seed)

    basic_context = BASIC_CONTEXT[book_code]
    char_registry.seed_narrator(basic_context["narrator_id"], basic_context["narrator_name"])

    identity_log = new_identity_log()
    entries_out = []
    relations_out = []
    mentions_out = []
    parse_errors = []

    for i, row in df.iterrows():
        prior_entries_df = df.iloc[:i]
        state_summary = build_state_summary(char_registry, loc_registry, org_registry, sig_event_registry)
        result = call_llm_for_entry(row, prior_entries_df, state_summary, basic_context)

        if "_parse_error" in result:
            parse_errors.append({"entry_id": row["entry_id"], **result})
            continue

        entry_date = result.get("entry_date_normalized")

        entries_out.append({
            "book": book_code,
            "entry_id": row["entry_id"],
            "chapter_id": row["chapter_id"],
            "entry_date_raw": result.get("entry_date_raw"),
            "entry_date": entry_date,
        })

        for rel in result.get("relations", []):
            relations_out.append(process_relation(
                rel, row["entry_id"], entry_date, char_registry, loc_registry, org_registry, sig_event_registry
            ))

        for m in result.get("mentions", []):
            eid, elabel = resolve_entity(m.get("surface_form"), m.get("type"),
                                          char_registry, loc_registry, org_registry, sig_event_registry, row["entry_id"])
            mentions_out.append({"book": book_code, "entry_id": row["entry_id"],
                                  "entity_id": eid, "entity_label": elabel, "entity_type": m.get("type")})

        for d in result.get("identity_descriptors", []):
            char_id, char_label = resolve_entity(d.get("character"), "character", char_registry, loc_registry,
                                                  org_registry, sig_event_registry, row["entry_id"])
            log_identity_descriptor(identity_log, char_id, char_label, d.get("raw_text"),
                                     d.get("attributed_by", "unspecified"), row["entry_id"])

        for u in result.get("unlisted_significant_events", []):
            sig_event_registry.unmatched_candidates.append({
                "label": u.get("raw_text"), "entry_id": row["entry_id"], "reason": u.get("reason", "")
            })

    return {
        "entries": entries_out,
        "relations": relations_out,
        "mentions": mentions_out,
        "identity_descriptors": identity_log,
        "parse_errors": parse_errors,
        "char_registry": char_registry,
        "loc_registry": loc_registry,
        "org_registry": org_registry,
        "sig_event_registry": sig_event_registry,
    }



## 13. Presence-over-time (map feed) & export

`build_presence_table` fills forward each character's `last_known_location`
across entries where it wasn't stated this entry — exactly the carry-forward
logic from the previous notebook, but generalized to every character in the
registry, not just the narrator, since that's what a multi-character map
needs. It's deliberately a *separate*, deterministic, auditable pass over
the already-extracted relations, not something baked into the LLM step.

This is a first cut: it fills forward based on the *last relation touching
that character*, in entry order, which is enough to try the map out, but
doesn't yet handle characters who are never mentioned in a given book's
back half, or overlapping/simultaneous timelines across characters — flag
those as things to look at once you see this run.


In [25]:

def build_presence_table(entries: list, char_registry: CharacterRegistry) -> list:
    entry_order = [e["entry_id"] for e in entries]
    entry_date_by_id = {e["entry_id"]: e["entry_date"] for e in entries}

    last_known = {cid: c.last_known_location for cid, c in char_registry._by_id.items()}
    stated_entry = {cid: c.last_location_stated_entry for cid, c in char_registry._by_id.items()}

    # Walk forward, carrying the *most recently stated as of this entry* location,
    # rather than just taking the registry's final state (which would leak future
    # knowledge backwards).
    presence = []
    running_location = {cid: None for cid in char_registry._by_id}
    running_stated_at = {cid: None for cid in char_registry._by_id}

    # Re-derive from relations to avoid leaking the registry's *final* state into
    # earlier entries — recompute walk-forward here rather than trusting the
    # registry snapshot above (kept above only as an end-state sanity check).
    return {
        "note": "Populate this from RELATIONS_OUT (located_in / travels_to) walked forward "
                "in entry order per character_id, once you've inspected the pilot output — "
                "left as a stub since the exact map granularity you want (per-entry vs. "
                "per-date, one point vs. a path) is worth deciding after seeing real data.",
        "entry_order": entry_order,
    }


def export_book(book_code: str, result: dict):
    with open(OUT_DIR / f"{book_code}_entries.json", "w") as f:
        json.dump(result["entries"], f, indent=2)
    with open(OUT_DIR / f"{book_code}_relations.json", "w") as f:
        json.dump(result["relations"], f, indent=2)
    with open(OUT_DIR / f"{book_code}_mentions.json", "w") as f:
        json.dump(result["mentions"], f, indent=2)
    with open(OUT_DIR / f"{book_code}_identity_descriptors.json", "w") as f:
        json.dump(result["identity_descriptors"], f, indent=2)

    with open(REGISTRY_DIR / f"{book_code}_characters.json", "w") as f:
        json.dump(result["char_registry"].export(), f, indent=2)
    with open(REGISTRY_DIR / f"{book_code}_locations.json", "w") as f:
        json.dump(result["loc_registry"].export(), f, indent=2)
    with open(REGISTRY_DIR / f"{book_code}_organizations.json", "w") as f:
        json.dump(result["org_registry"].export(), f, indent=2)
    with open(REGISTRY_DIR / f"{book_code}_significant_events.json", "w") as f:
        json.dump(result["sig_event_registry"].export(), f, indent=2)

    with open(REVIEW_DIR / f"{book_code}_pending_characters.json", "w") as f:
        json.dump(result["char_registry"].pending_review, f, indent=2)
    with open(REVIEW_DIR / f"{book_code}_pending_locations.json", "w") as f:
        json.dump(result["loc_registry"].pending_review, f, indent=2)
    with open(REVIEW_DIR / f"{book_code}_pending_organizations.json", "w") as f:
        json.dump(result["org_registry"].pending_review, f, indent=2)
    with open(REVIEW_DIR / f"{book_code}_unmatched_significant_events.json", "w") as f:
        json.dump(result["sig_event_registry"].unmatched_candidates, f, indent=2)
    if result["parse_errors"]:
        with open(REVIEW_DIR / f"{book_code}_parse_errors.json", "w") as f:
            json.dump(result["parse_errors"], f, indent=2)

    n_interpretive = sum(1 for r in result["relations"] if r["interpretive"])
    print(f"[{book_code}] {len(result['entries'])} entries processed, "
          f"{len(result['relations'])} relations extracted ({n_interpretive} interpretive), "
          f"{len(result['identity_descriptors'])} identity descriptors logged.")
    print(f"  Flagged for review: {len(result['char_registry'].pending_review)} characters, "
          f"{len(result['loc_registry'].pending_review)} locations, "
          f"{len(result['org_registry'].pending_review)} organizations, "
          f"{len(result['sig_event_registry'].unmatched_candidates)} unmatched significant events.")
    if result["parse_errors"]:
        print(f"  {len(result['parse_errors'])} entries failed to parse — see {book_code}_parse_errors.json")



## 14. Running this on *A Sea of Sorrows*

**Data you need to provide:** `./data/SofS.csv` with columns `book,
chapter_id, entry_id, date, entry_text` — one row per diary entry, in
reading order. `chapter_id` should contain a chapter number (e.g. `ch1`),
`entry_id` should end in `e<number>` (e.g. `SofS_ch1_e1`) since the loader
sorts on those patterns; adjust `load_book`'s regexes if your actual IDs
don't match that shape. `date` can just be whatever raw string you have per
entry (or even left blank) — it's not load-bearing, since the LLM
re-extracts `entry_date_raw`/`entry_date_normalized` from the entry text
itself.

**Environment:** `ANTHROPIC_API_KEY` set, `spacy` + `en_core_web_sm` (or
`en_core_web_trf`) installed, `anthropic` and `pandas` installed.

**Order of operations:**
1. Run cell in §1 to load the CSV.
2. Run §2 to get the NER candidate lists, inspect `output/candidate_lists/SofS_candidates.json`.
3. Run §3's `suggest_canonical_entities` (or hand-edit) to produce `output/candidate_lists/SofS_reviewed.json`. At minimum this should contain Michael, Patrick, the parents, and any locations named early (Ireland, Cork or wherever they depart from, Grosse Île, Quebec).
4. `reviewed = load_reviewed_entities("SofS")`
5. `book_df = load_book("SofS")`
6. `result = process_book("SofS", book_df, reviewed)`
7. `export_book("SofS", result)`
8. Look at everything under `output/for_review/` first — that's where the pipeline is telling you it wasn't sure. Then spot-check `output/SofS_relations.json` against a few entries you know well by eye, the way you'd have done with the old debug/QA setup, just without the formal grounding-check machinery this time.

**Things worth watching for on this first run specifically**, given the book's content: the mid-book family deaths mean `stops_living_with` and `grieves_for` should start appearing once that happens — check whether the model handles a death correctly (is it `grieves_for`, or does it wrongly keep emitting `located_in`/`lives_with` for a character who has died?). Also worth checking: does `travels_to` correctly capture the *hoped-for* destination framing (per the prompt's guidance) rather than only firing on literal arrivals — that distinction was untested until you run real text through it.


In [28]:
# Process SofS

result = process_book("SofS", book_df, reviewed)
export_book("SofS", result)

TypeError: Messages.create() got an unexpected keyword argument 'temperature'